# GLM-5.3-Flash-Uncensored-FP8 on Kaggle TPU v5e-8

Custom pure-JAX serving engine for the 321B-param FP8 MoE that cannot fit
in HBM: dense weights TP-sharded across 8 chips, all 288x42 routed experts
cold in the 330 GB host RAM, 8-expert hot banks per chip/layer in HBM.

- Weights: `zai-org/GLM-5.3-Flash` (ungated, identical geometry to the
  orcarouter ab-literated FP8 variant)
- Architecture: 34 KDA linear-attention + 11 DSA/MLA layers, mHC
  hyper-connections, sigmoid/noaux_tc router, NoPE (zero rope)
- v1 simplifications: DSA lightning-indexer skipped (full attention over
  the 512-d latent cache), MTP head + vision tower never downloaded
- API: OpenAI-compatible on :8080 + cloudflared public tunnel (URL printed
  below, rotates each session; same API key as the GPU notebook)
- Every engine module was validated on 8 simulated CPU devices before this
  push: prefill/decode bit-deterministic, hot-bank refresh exact

In [ ]:
import os, sys, time

print("checking TPU...")
import jax
print("jax", jax.__version__)
devs = jax.devices()
print("devices:", devs)
assert len(devs) == 8, f"expected 8 TPU devices, got {len(devs)}"
os.makedirs("/kaggle/tmp/glmtpu", exist_ok=True)


In [ ]:
# ---- engine modules: pull from GitHub, fall back to embedded copies ----
# (embedded MODS below stay in sync with the repo; git wins when reachable)
import os, sys, subprocess

# ---- weights source: orcarouter ab-literated (HF-gated) if a token exists ----
# Token priority: Kaggle secret "HF_TOKEN" -> env HF_TOKEN -> none (zai base)
HF_TOKEN = None
try:
    from kaggle_web_client import UserSecretClient
    HF_TOKEN = UserSecretClient().get_secret("HF_TOKEN")
    print("HF token: loaded from Kaggle secret")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN") or None
    if HF_TOKEN:
        print("HF token: from env")

if HF_TOKEN:
    os.environ["GLM_REPO"] = "orcarouter/GLM-5.3-Flash-Unc-ensored-FP8"
else:
    os.environ.setdefault("GLM_REPO", "zai-org/GLM-5.3-Flash")
print("weights repo:", os.environ["GLM_REPO"])

os.makedirs("/kaggle/tmp", exist_ok=True)
try:
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/lmaohhh/tpu-glm.git",
                    "/kaggle/tmp/tpu-glm"], check=True, timeout=120)
    sys.path.insert(0, "/kaggle/tmp/tpu-glm/src")
    print("engine: from GitHub @", subprocess.run(
        ["git", "-C", "/kaggle/tmp/tpu-glm", "rev-parse", "--short", "HEAD"],
        capture_output=True, text=True).stdout.strip())
except Exception as e:
    print("git pull failed -> using embedded modules:", e)

if "glmtpu" not in sys.modules:
    MODS = {}
    MODS["fp8.py"] = '''"""FP8 (e4m3fn) 128x128-blockwise dequantization.

numpy LUT for host-side dequant (dense/small weights -> BF16 at load time);
JAX in-graph version only for routed-expert banks (whole experts, row0=col0=0).
"""
from __future__ import annotations

import numpy as np

BLOCK = 128


def _build_lut() -> np.ndarray:
    lut = np.zeros(256, dtype=np.float32)
    for u in range(256):
        sign = -1.0 if (u & 0x80) else 1.0
        exp = (u >> 3) & 0xF
        man = u & 0x7
        if exp == 0:
            val = (2.0 ** -6) * (man / 8.0)          # subnormal
        elif (u & 0x7F) == 0x7F:
            val = 0.0                                 # NaN -> 0 (not in weights)
        else:
            val = (2.0 ** (exp - 7)) * (1.0 + man / 8.0)
        lut[u] = sign * val
    return lut


F8_LUT = _build_lut()


def dequant_np(fp8_u8: np.ndarray, scale_inv: np.ndarray) -> np.ndarray:
    """uint8 [R,C] + f32 scale_inv [ceil(R/128), ceil(C/128)] -> f32 [R,C].
    W = fp8 * scale_inv (scale_inv holds the INVERSE scale)."""
    R, C = fp8_u8.shape
    vals = F8_LUT[fp8_u8]                               # [R,C] f32 LUT gather
    out = np.empty((R, C), dtype=np.float32)
    for i in range(0, R, BLOCK):
        for j in range(0, C, BLOCK):
            out[i:i + BLOCK, j:j + BLOCK] = (
                vals[i:i + BLOCK, j:j + BLOCK] * scale_inv[i // BLOCK, j // BLOCK])
    return out


def fp8_matmul(x, fp8_u8, scale_inv, block=BLOCK):
    """JAX in-graph: x [.., in] bf16 @ dequant(fp8 [out, in])^T -> [.., out].
    Only used for whole-expert banks (global row/col both start at 0)."""
    import jax.numpy as jnp
    from jax import lax
    out_dim, in_dim = fp8_u8.shape
    fp8 = lax.bitcast_convert_type(fp8_u8, jnp.float8_e4m3fn).astype(jnp.float32)
    row = jnp.arange(out_dim) // block
    col = jnp.arange(in_dim) // block
    s = scale_inv[row[:, None], col[None, :]]           # [out, in] f32
    w = (fp8 * s).astype(jnp.bfloat16)
    return x @ w.T
'''
    MODS["config.py"] = '''"""GLM-5.3-Flash (glm5_next) serving config.

Derived from zai-org/GLM-5.3-Flash config.json (transformers 5.16.0) and
verified against real safetensors shard headers.  The orcarouter ab-literated
FP8 variant shares identical geometry (only weights differ).
"""
from __future__ import annotations

import dataclasses
import json
from typing import Optional


@dataclasses.dataclass
class GlmConfig:
    # ---- layer geometry (real values from config.json) ----
    hidden_size: int = 4096
    vocab_size: int = 154880
    n_layers: int = 45              # main stack 0..44; layer 45 (MTP) never loaded
    n_kda_heads: int = 64
    kda_head_dim: int = 128
    conv_kernel: int = 4
    gate_lower_bound: float = -5.0
    dsa_heads: int = 64
    q_lora_rank: int = 1536
    kv_lora_rank: int = 512
    qk_nope_head_dim: int = 256
    v_head_dim: int = 256
    qk_rope_head_dim: int = 0      # NoPE: no rope anywhere in the text stack
    dense_inter: int = 12288       # first 3 layers
    moe_inter: int = 2048
    n_experts: int = 288
    top_k: int = 8
    n_shared_experts: int = 1
    routed_scaling_factor: float = 2.5
    swiglu_limit: float = 10.0
    hc_mult: int = 4
    hc_sinkhorn_iters: int = 20
    hc_eps: float = 1e-6
    rms_norm_eps: float = 1e-5
    eos_ids: tuple = (154820, 154827, 154829)
    # serving
    max_ctx: int = 32768
    prefill_chunk: int = 256
    n_slots: int = 64           # decode bank slots per chip (>= top_k)
    n_passes: int = 1           # reserved

    kda_layers: tuple = ()
    dsa_layers: tuple = ()
    moe_layers: tuple = ()
    dense_mlp_layers: tuple = ()

    @staticmethod
    def real() -> "GlmConfig":
        c = GlmConfig()
        c.kda_layers = tuple(l for l in range(45) if l % 4 != 3)
        c.dsa_layers = tuple(range(3, 45, 4))
        c.dense_mlp_layers = (0, 1, 2)
        c.moe_layers = tuple(range(3, 45))
        return c

    @staticmethod
    def tiny() -> "GlmConfig":
        c = GlmConfig(
            hidden_size=128, vocab_size=512, n_layers=4,
            n_kda_heads=8, kda_head_dim=16, conv_kernel=4,
            dsa_heads=8, q_lora_rank=32, kv_lora_rank=16,
            qk_nope_head_dim=8, v_head_dim=8,
            dense_inter=128, moe_inter=64, n_experts=16, top_k=2,
            hc_mult=2, hc_sinkhorn_iters=3,
            max_ctx=256, prefill_chunk=64, n_slots=2, n_passes=1,
        )
        c.kda_layers = (0, 1, 2)
        c.dsa_layers = (3,)
        c.dense_mlp_layers = (0, 1)
        c.moe_layers = (2, 3)
        return c

    def is_kda(self, l: int) -> bool:
        return l in self.kda_layers

    def is_moe(self, l: int) -> bool:
        return l in self.moe_layers

    def to_json(self) -> str:
        d = dataclasses.asdict(self)
        d["eos_ids"] = list(self.eos_ids)
        return json.dumps(d, indent=1)

    @staticmethod
    def from_json(s: str) -> "GlmConfig":
        d = json.loads(s)
        d["eos_ids"] = tuple(d["eos_ids"])
        return GlmConfig(**d)
'''
    MODS["layers.py"] = '''"""glm5_next per-layer math in pure JAX — FINAL.

Mirrored 1:1 from transformers v5.16.0 modeling_glm5_next.py (line refs).
Every function runs on ONE chip with LOCAL shards, inside jax.pmap(axis_name
='tp').  Collectives: lax.psum(..., 'tp') for column-parallel outputs.

Sharding:
  KDA (34 layers)   heads 64 -> 8 per chip: q/k/v_proj, f_b, g_b, dt_bias
                    row-sharded; o_proj col-sharded + psum; A_log/b_proj/
                    conv channel-sharded; f_a/g_a/o_norm replicated.
  DSA (11 layers)   q_b/kv_b row-sharded by head; o_proj col-sharded + psum;
                    q_a/kv_a/norms replicated; latent cache replicated.
  MLP dense/shared  FP8 Megatron pairing: gate/up row-sharded, down
                    col-sharded + psum; in-graph dequant.
  MoE routed        per-chip expert banks (FP8, static n_slots); every chip
                    applies its resident experts to ALL tokens with router
                    coeff masks; psum.  Exact for any bank contents (misses
                    contribute 0; host guarantees coverage before the call).
  Router / hc / norms / embed(host) / final_ln: replicated.
"""
from __future__ import annotations

import jax
import jax.numpy as jnp
from jax import lax

from .fp8 import fp8_matmul

# ===========================================================================
# basic ops
# ===========================================================================


def rms_norm(x, w, eps):
    x32 = x.astype(jnp.float32)
    x32 = x32 * lax.rsqrt(jnp.mean(x32 * x32, axis=-1, keepdims=True) + eps)
    return (w.astype(jnp.float32) * x32).astype(x.dtype)


def rms_norm_no_w(x, eps):
    x32 = x.astype(jnp.float32)
    return (x32 * lax.rsqrt(jnp.mean(x32 * x32, axis=-1, keepdims=True) + eps)).astype(x.dtype)


def l2norm(x, eps=1e-6):
    # modeling lines 418-426 (FLA style: divide by sqrt(sum^2 + eps))
    return x / jnp.sqrt(jnp.sum(x * x, axis=-1, keepdims=True) + eps)


def swiglu_clamped(gate, up, limit):
    # modeling lines 99-105 / 138-143
    gate = jnp.clip(gate, None, limit)
    up = jnp.clip(up, -limit, limit)
    return jax.nn.silu(gate.astype(jnp.float32)).astype(gate.dtype) * up


# ===========================================================================
# hyper-connections (modeling lines 220-303, 1318-1329)
# ===========================================================================

def hc_site(hc, streams, cfg):
    """hc = {"fn": [mix, H*D] bf16, "base": [mix] f32, "scale": [3] f32}
    (replicated).  streams [B,S,H,D] -> (post [B,S,H] f32, comb [B,S,H,H] f32
    indexed [b,s,j,h], collapsed [B,S,D] bf16)."""
    B, S, H, D = streams.shape
    flat = streams.reshape(B, S, H * D).astype(jnp.float32)
    flat = rms_norm_no_w(flat, cfg.rms_norm_eps)          # input_norm
    logits = flat @ hc["fn"].astype(jnp.float32).T        # [B,S,(2+H)*H]
    pre_w, post_w, comb_w = jnp.split(logits, [H, 2 * H], axis=-1)
    pre_b, post_b, comb_b = jnp.split(hc["base"].astype(jnp.float32), [H, 2 * H])
    pre_s, post_s, comb_s = hc["scale"][0], hc["scale"][1], hc["scale"][2]

    pre = jax.nn.sigmoid(pre_w * pre_s + pre_b) + cfg.hc_eps   # line 284
    post = 2.0 * jax.nn.sigmoid(post_w * post_s + post_b)      # line 285
    cl = comb_w.reshape(B, S, H, H) * comb_s + comb_b.reshape(H, H)
    comb = jax.nn.softmax(cl, axis=-1) + cfg.hc_eps            # line 287
    comb = comb / (jnp.sum(comb, axis=-2, keepdims=True) + cfg.hc_eps)  # 288
    for _ in range(cfg.hc_sinkhorn_iters - 1):                 # 289-291
        comb = comb / (jnp.sum(comb, axis=-1, keepdims=True) + cfg.hc_eps)
        comb = comb / (jnp.sum(comb, axis=-2, keepdims=True) + cfg.hc_eps)
    collapsed = jnp.sum(pre[..., None] * streams, axis=2)      # line 295
    return post, comb, collapsed.astype(streams.dtype)


def hc_apply(post, comb, sub_out, streams):
    """out[h] = post[h]*sub_out + sum_j comb[j,h]*stream[j]  (lines 1318-1320)."""
    term_a = post[..., None].astype(sub_out.dtype) * sub_out[..., None, :]
    term_b = jnp.einsum("bsjh,bsjd->bshd", comb.astype(streams.dtype), streams)
    return term_a + term_b


# ===========================================================================
# KDA (modeling lines 306-735)
# ===========================================================================

def _dwconv_silu(u, w, cs, K):
    """Depthwise causal conv + silu.  u [B,S,C] bf16, w [C,K] bf16,
    cs [B,C,K-1] bf16 (channels-FIRST).  up = concat(cs, u^T) on time:
    out[t] = sum_i w[:,i] * up[:, t+i]  (lines 376-415).
    Zero left-pad == zero conv state, so pad rows never affect real rows."""
    S = u.shape[1]
    up = jnp.concatenate([cs, u.transpose(0, 2, 1)], axis=2)   # [B,C,S+K-1]
    acc = jnp.zeros((u.shape[0], u.shape[2], S), dtype=jnp.float32)
    for i in range(K):
        acc = acc + up[:, :, i:i + S].astype(jnp.float32) * w[:, i].astype(jnp.float32)[None, :, None]
    out = jax.nn.silu(acc).transpose(0, 2, 1)                  # [B,S,C]
    new_cs = up[:, :, S:]                                       # [B,C,K-1]
    return out.astype(u.dtype), new_cs


def kda_core(p, x, rec, conv, valid, cfg):
    """x [B,S,D] bf16 (zeroed at pads), valid [B,S] f32 (1 real / 0 pad).
    rec [B,nh_l,hd,hd] f32 LOCAL; conv [B,3*qkvd_l,K-1] bf16 LOCAL.
    Returns (out [B,S,D] bf16 psum'ed, rec', conv')."""
    B, S, D = x.shape
    nh, hd = rec.shape[1], rec.shape[2]                   # LOCAL heads
    qkvd = nh * hd
    K = cfg.conv_kernel

    q = x @ p["q_proj"].T
    k = x @ p["k_proj"].T
    v = x @ p["v_proj"].T
    q, csq = _dwconv_silu(q, p["q_conv"], conv[:, :qkvd], K)
    k, csk = _dwconv_silu(k, p["k_conv"], conv[:, qkvd:2 * qkvd], K)
    v, csv = _dwconv_silu(v, p["v_conv"], conv[:, 2 * qkvd:], K)

    q = l2norm(q.reshape(B, S, nh, hd).astype(jnp.float32))
    k = l2norm(k.reshape(B, S, nh, hd).astype(jnp.float32))
    v = v.reshape(B, S, nh, hd).astype(jnp.float32)
    q = q * (hd ** -0.5)                                  # after l2norm (453-454)

    # forget gate (lines 306-336): g = lb * sigmoid(exp(A_log)*(f_b f_a x + dt_bias))
    h1 = x @ p["f_a"].T                                   # [B,S,hd] replicated
    fg = (h1 @ p["f_b"].T).astype(jnp.float32) + p["dt_bias"].astype(jnp.float32)
    fg = fg.reshape(B, S, nh, hd)
    decay = jnp.exp(p["A_log"].astype(jnp.float32)).reshape(1, 1, nh, 1)
    g = cfg.gate_lower_bound * jax.nn.sigmoid(decay * fg)  # in (lb, 0)

    beta = jax.nn.sigmoid((x @ p["b_proj"].T).astype(jnp.float32))  # [B,S,nh]

    def step(state, inp):
        qi, ki, vi, gi, bi, val = inp
        s = state * jnp.exp(gi)[..., None]                # line 473
        kv_mem = jnp.sum(s * ki[..., None, :], axis=-2)   # line 474
        delta = (vi - kv_mem) * bi[..., None]             # line 475
        s = s + ki[..., :, None] * delta[..., None, :]    # line 477
        out = jnp.sum(s * qi[..., None, :], axis=-2)      # line 478
        s = jnp.where(val > 0.5, s, state)                # pads leave state
        out = out * val                                    # pads output 0
        return s, out

    # scan over the sequence axis: transpose to [S, B, ...]
    def t(a):
        return a.transpose(1, 0, *range(2, a.ndim))
    rec, core_t = lax.scan(step, rec, (t(q), t(k), t(v), t(g),
                                       t(beta) if beta.ndim > 2 else beta.transpose(1, 0),
                                       t(valid)))
    core = core_t.transpose(1, 0, 2, 3)                   # [B,S,nh,hd]

    # gated output norm (lines 341-360) + col-parallel o_proj
    og = ((x @ p["g_a"].T) @ p["g_b"].T).reshape(B, S, nh, hd)
    n32 = core.astype(jnp.float32)                       # [B,S,nh,hd]
    n32 = n32 * lax.rsqrt(jnp.mean(n32 * n32, axis=-1, keepdims=True) + cfg.rms_norm_eps)
    n32 = p["o_norm"].astype(jnp.float32) * n32
    n32 = n32 * jax.nn.sigmoid(og.astype(jnp.float32))
    coreb = n32.astype(x.dtype).reshape(B, S, qkvd)
    out = coreb @ p["o_proj"].T                           # col-sharded partial
    out = lax.psum(out, "tp")
    new_conv = jnp.concatenate([csq, csk, csv], axis=1)
    return out, rec, new_conv


# ===========================================================================
# DSA / MLA layer: full causal attention over the replicated latent cache with
# absorbed W_UK / W_UV (modeling lines 1066-1258).  v1: indexer skipped
# (documented quality caveat; exactly correct causal masking retained).
# ===========================================================================

def dsa_core(p, x, kv, bitmap, cache_len, chunk_valid, cfg):
    """x [B,S,D] bf16; kv [B,max_ctx,dkv] bf16 replicated (pre-chunk state);
    bitmap [B,max_ctx] f32 (1 = valid key), updated with chunk_valid at
    [cache_len : cache_len+S] inside; cache_len traced i32; chunk_valid
    [B,S] f32.  Returns (out, kv', bitmap')."""
    B, S, D = x.shape
    H = p["q_b"].shape[0] // cfg.qk_nope_head_dim          # LOCAL heads
    dk, dv, dkv = cfg.qk_nope_head_dim, cfg.v_head_dim, cfg.kv_lora_rank
    T = kv.shape[1]

    q_res = rms_norm(x @ p["q_a"].T, p["q_a_ln"], cfg.rms_norm_eps)   # [B,S,qlora]
    q = (q_res @ p["q_b"].T).reshape(B, S, H, dk)
    c = rms_norm(x @ p["kv_a"].T, p["kv_a_ln"], cfg.rms_norm_eps)     # [B,S,dkv]

    w = p["kv_b"].reshape(H, dk + dv, dkv)
    w_uk = w[:, :dk]
    w_uv = w[:, dk:]

    # write chunk latents into the cache at [cache_len : cache_len+S]
    c = jnp.where(chunk_valid[..., None] > 0.5, c, 0.0).astype(kv.dtype)
    kv = lax.dynamic_update_slice(kv, c, (0, cache_len, 0))
    bitmap = lax.dynamic_update_slice(
        bitmap, chunk_valid.astype(bitmap.dtype), (0, cache_len))

    # absorbed query: q' = W_UK^T q -> [B,S,H,dkv]
    qp = jnp.einsum("bshd,hdk->bshk",
                    q.astype(jnp.float32), w_uk.astype(jnp.float32))
    # full-cache attention with causal + bitmap masks
    qpos = cache_len + jnp.arange(S)
    kpos = jnp.arange(T)
    causal = kpos[None, :] <= qpos[:, None]               # [S,T]
    mask = causal[None, :, :] & (bitmap[:, :T] > 0.5)     # [B,S,T]
    scores = jnp.einsum("bshk,btk->bhst",
                        qp.astype(kv.dtype), kv) * (dk ** -0.5)
    scores = scores.astype(jnp.float32)
    scores = jnp.where(mask[:, None, :, :], scores, -1e30)
    probs = jax.nn.softmax(scores, axis=-1)
    U = jnp.einsum("bhst,btk->bshk", probs.astype(kv.dtype), kv)
    vo = jnp.einsum("bshk,hdk->bshd",
                    U.astype(jnp.float32), w_uv.astype(jnp.float32))
    out = vo.reshape(B, S, H * dv).astype(x.dtype) @ p["o_proj"].T
    out = lax.psum(out, "tp")
    return out, kv, bitmap


# ===========================================================================
# MLPs
# ===========================================================================

def dense_mlp_core(p, h, cfg):
    """BF16 Megatron pairing (weights host-dequantized at load):
    gate/up row-sharded, down col-sharded.  Returns [B,S,D] bf16 (psum'ed)."""
    g = h @ p["g"].T
    u = h @ p["u"].T
    hid = swiglu_clamped(g, u, cfg.swiglu_limit)
    out = hid @ p["d"].T
    return lax.psum(out, "tp")


def moe_router(gate_w, e_bias, h, cfg):
    """Replicated router (lines 146-184; n_group=topk_group=1 => identity).
    h [B,S,D] -> (weights [B,S,K] f32, ids [B,S,K] i32)."""
    logits = h.astype(jnp.float32) @ gate_w.astype(jnp.float32).T   # [B,S,E]
    scores = jax.nn.sigmoid(logits)                        # line 162
    choice = scores + e_bias.astype(jnp.float32)           # line 163 (noaux_tc)
    ids = lax.top_k(choice, cfg.top_k)[1]                  # line 178
    w = jnp.take_along_axis(scores, ids, axis=-1)          # line 179
    w = w / (jnp.sum(w, axis=-1, keepdims=True) + 1e-20)   # line 181-182
    w = w * cfg.routed_scaling_factor                      # line 183
    return w, ids


def moe_bank_core(bank, h, w, ids, cfg):
    """Per-chip expert bank application.  bank: {"gu": [n,2I,D] u8,
    "gu_s": [n, ceil(2I/128), ceil(D/128)] f32, "d": [n,D,I] u8, "d_s": ...,
    "ids": [n] i32}.  Slot ids of -1 are padding (never match).  Every chip
    applies its resident experts to ALL tokens; psum makes the sum exact
    given host-side coverage of the union of routed ids."""
    B, S, D = h.shape
    n = bank["ids"].shape[0]
    out = jnp.zeros((B, S, D), dtype=jnp.float32)
    for s in range(n):
        e = bank["ids"][s]
        match = (ids == e).astype(jnp.float32)             # [B,S,K]
        coeff = jnp.sum(match * w, axis=-1)                # [B,S]
        gu = fp8_matmul(h, bank["gu"][s], bank["gu_s"][s])  # [B,S,2I]
        gate, up = jnp.split(gu, 2, axis=-1)
        hid = swiglu_clamped(gate, up, cfg.swiglu_limit)   # [B,S,I]
        y = fp8_matmul(hid, bank["d"][s], bank["d_s"][s])  # [B,S,D]
        out = out + y.astype(jnp.float32) * coeff[..., None]
    return lax.psum(out.astype(h.dtype), "tp")


# ===========================================================================
# site wrappers (one pmap'ed call per attention/FFN site)
# ===========================================================================

def attn_site_kda(p, streams, valid, rec, conv, cfg):
    """p['attn_hc'], p['kda'], p['input_ln'].  streams [B,S,H,D] bf16;
    valid [B,S] f32; rec/conv LOCAL per chip.  Returns (streams', rec', conv').

    Pad-row invariant: pad rows of streams stay exactly zero."""
    post, comb, collapsed = hc_site(p["attn_hc"], streams, cfg)
    h = rms_norm(collapsed, p["input_ln"], cfg.rms_norm_eps)
    out, rec2, conv2 = kda_core(p["kda"], h, rec, conv, valid, cfg)
    streams2 = hc_apply(post, comb, out, streams)
    streams2 = streams2 * valid[..., None, None].astype(streams2.dtype)
    return streams2, rec2, conv2


def attn_site_dsa(p, streams, valid, kv, bitmap, cache_len, cfg):
    """p['attn_hc'], p['dsa'], p['input_ln'].  kv/bitmap LOCAL replicas
    (identical on all chips); cache_len traced i32 scalar."""
    post, comb, collapsed = hc_site(p["attn_hc"], streams, cfg)
    h = rms_norm(collapsed, p["input_ln"], cfg.rms_norm_eps)
    out, kv2, bm2 = dsa_core(p["dsa"], h, kv, bitmap, cache_len, valid, cfg)
    streams2 = hc_apply(post, comb, out, streams)
    streams2 = streams2 * valid[..., None, None].astype(streams2.dtype)
    return streams2, kv2, bm2


def ffn_site_dense(p, streams, cfg):
    """p['ffn_hc'], p['mlp'], p['post_ln']."""
    post, comb, collapsed = hc_site(p["ffn_hc"], streams, cfg)
    h = rms_norm(collapsed, p["post_ln"], cfg.rms_norm_eps)
    y = dense_mlp_core(p["mlp"], h, cfg)
    streams2 = hc_apply(post, comb, y, streams)
    return streams2


def ffn_site_moe(p, bank, streams, cfg, collect_router=False):
    """p['ffn_hc'], p['moe'] (router+shared), bank per chip."""
    post, comb, collapsed = hc_site(p["ffn_hc"], streams, cfg)
    h = rms_norm(collapsed, p["post_ln"], cfg.rms_norm_eps)
    w, ids = moe_router(p["moe"]["gate_w"], p["moe"]["e_bias"], h, cfg)
    sh = dense_mlp_core(p["moe"]["sh"], h, cfg)
    moe = moe_bank_core(bank, h, w, ids, cfg)
    y = sh + moe
    streams2 = hc_apply(post, comb, y, streams)
    if collect_router:
        return streams2, ids
    return streams2
'''
    MODS["params.py"] = '''"""Build params_by_chip for the Runner: FAKE weights (local structural test)
or REAL safetensors shards (loader_real.py mirrors this layout exactly).

params_by_chip[c] = {
    l: {
      "attn_hc": {"fn","base","scale"},      # replicated
      "ffn_hc": {...},
      "input_ln": [D], "post_ln": [D],
      "kda": {...} | None,                    # per-chip shards
      "dsa": {...} | None,
      "moe": {"gate_w","e_bias","sh":{...}} | None,   # sh = shared experts, BF16
      "mlp": {...} | None,                    # dense MLP, BF16
    },
    "final_ln": [D],
}

All BF16 tensors are stored as float32 numpy (bf16 values, f32 container) so
numpy can slice them; JAX casts to bf16 at device_put.

Also returns (embed_f32 [V,D], lm_head_f32 [V,D], expert_host).
expert_host[(layer, e)] = {"gu": u8 [2I,D], "gu_s": f32 grid, "d": u8 [D,I],
                           "d_s": f32 grid}  (gate|up fused on axis 0).
"""
from __future__ import annotations

import numpy as np

import jax.numpy as jnp

from .config import GlmConfig

BLOCK = 128


def _ceil(n, d):
    return -(-n // d)


def _slice_rows(w, d, c):
    chunk = _ceil(w.shape[0], d)
    return w[c * chunk:(c + 1) * chunk]


def _slice_cols(w, d, c):
    chunk = _ceil(w.shape[1], d)
    return w[:, c * chunk:(c + 1) * chunk]


def make_fake(cfg: GlmConfig, d: int, seed: int = 0):
    rng = np.random.default_rng(seed)
    D = cfg.hidden_size
    H = cfg.hc_mult
    mix = (2 + H) * H

    def bf(shape, scale=0.02):
        a = (rng.standard_normal(shape) * scale).astype(np.float32)
        return np.asarray(jnp.asarray(a).astype(jnp.bfloat16).astype(jnp.float32))

    def f32(shape, scale=0.02):
        return (rng.standard_normal(shape) * scale).astype(np.float32)

    def u8_pair(rows, cols):
        """FP8 weight [rows, cols] + scale_inv grid (for expert banks)."""
        from .fp8 import F8_LUT
        w = (rng.standard_normal((rows, cols)) * 0.05).astype(np.float32)
        s = (np.abs(rng.standard_normal((_ceil(rows, BLOCK), _ceil(cols, BLOCK))))
             * 0.01 + 0.005).astype(np.float32)
        scaled = np.empty_like(w)
        for i in range(s.shape[0]):
            for j in range(s.shape[1]):
                scaled[i * BLOCK:(i + 1) * BLOCK, j * BLOCK:(j + 1) * BLOCK] = \
                    w[i * BLOCK:(i + 1) * BLOCK, j * BLOCK:(j + 1) * BLOCK] / s[i, j]
        scaled = np.clip(scaled, -448.0, 448.0)
        sign = np.sign(scaled)
        a = np.abs(scaled)
        grid = np.sort(np.unique(np.abs(F8_LUT)))
        idx = np.clip(np.searchsorted(grid, a), 1, len(grid) - 1)
        left, right = grid[idx - 1], grid[idx]
        chosen = np.where(a - left <= right - a, left, right)
        q = np.zeros(scaled.shape, np.uint8)
        for g in np.unique(chosen):
            u_val = int(np.where(np.abs(F8_LUT) == g)[0][0])
            u = np.where(sign < 0, u_val | 0x80, u_val)
            q[chosen == g] = u[chosen == g].astype(np.uint8)
        return q, s

    nh = cfg.n_kda_heads
    hd = cfg.kda_head_dim
    qkvd = nh * hd
    K = cfg.conv_kernel
    E = cfg.n_experts
    I = cfg.moe_inter
    Hh = cfg.dsa_heads
    dk, dv, dkv = cfg.qk_nope_head_dim, cfg.v_head_dim, cfg.kv_lora_rank

    full = {}
    for l in range(cfg.n_layers):
        lay = {
            "attn_hc": {"fn": bf((mix, H * D)), "base": f32((mix,), 0.0),
                        "scale": np.array([1.0, 1.0, 1.0], np.float32)},
            "ffn_hc": {"fn": bf((mix, H * D)), "base": f32((mix,), 0.0),
                       "scale": np.array([1.0, 1.0, 1.0], np.float32)},
            "input_ln": np.ones(D, np.float32),
            "post_ln": np.ones(D, np.float32),
            "kda": None, "dsa": None, "moe": None, "mlp": None,
        }
        if cfg.is_kda(l):
            lay["kda"] = {
                "q_proj": bf((qkvd, D)), "k_proj": bf((qkvd, D)), "v_proj": bf((qkvd, D)),
                "q_conv": bf((qkvd, K)), "k_conv": bf((qkvd, K)), "v_conv": bf((qkvd, K)),
                "f_a": bf((hd, D)), "f_b": bf((qkvd, hd)),
                "dt_bias": f32((qkvd,), 0.001), "A_log": f32((nh,), 0.0),
                "b_proj": bf((nh, D)),
                "g_a": bf((hd, D)), "g_b": bf((qkvd, hd)),
                "o_norm": np.ones(hd, np.float32),
                "o_proj": bf((D, qkvd)),
            }
        else:
            lay["dsa"] = {
                "q_a": bf((cfg.q_lora_rank, D)),
                "q_a_ln": np.ones(cfg.q_lora_rank, np.float32),
                "q_b": bf((Hh * dk, cfg.q_lora_rank)),
                "kv_a": bf((dkv, D)), "kv_a_ln": np.ones(dkv, np.float32),
                "kv_b": bf((Hh * (dk + dv), dkv)),
                "o_proj": bf((D, Hh * dv)),
            }
        if cfg.is_moe(l):
            sh_g, _ = u8_pair(I, D)
            sh_u, _ = u8_pair(I, D)
            sh_d, _ = u8_pair(D, I)
            lay["moe"] = {
                "gate_w": bf((E, D)), "e_bias": f32((E,), 0.0),
                "sh": {"g": bf((I, D)), "u": bf((I, D)), "d": bf((D, I))},
            }
        else:
            lay["mlp"] = {"g": bf((cfg.dense_inter, D)),
                          "u": bf((cfg.dense_inter, D)),
                          "d": bf((D, cfg.dense_inter))}
        full[l] = lay

    params_by_chip = []
    for c in range(d):
        chip = {}
        for l in range(cfg.n_layers):
            lay = {
                "attn_hc": {k: v.copy() for k, v in full[l]["attn_hc"].items()},
                "ffn_hc": {k: v.copy() for k, v in full[l]["ffn_hc"].items()},
                "input_ln": full[l]["input_ln"].copy(),
                "post_ln": full[l]["post_ln"].copy(),
                "kda": None, "dsa": None, "moe": None, "mlp": None,
            }
            if cfg.is_kda(l):
                k = full[l]["kda"]
                lay["kda"] = {
                    "q_proj": _slice_rows(k["q_proj"], d, c),
                    "k_proj": _slice_rows(k["k_proj"], d, c),
                    "v_proj": _slice_rows(k["v_proj"], d, c),
                    "q_conv": _slice_rows(k["q_conv"], d, c),
                    "k_conv": _slice_rows(k["k_conv"], d, c),
                    "v_conv": _slice_rows(k["v_conv"], d, c),
                    "f_a": k["f_a"],                       # replicated
                    "f_b": _slice_rows(k["f_b"], d, c),
                    "dt_bias": _slice_rows(k["dt_bias"].reshape(-1, 1), d, c).reshape(-1),
                    "A_log": _slice_rows(k["A_log"].reshape(-1, 1), d, c).reshape(-1),
                    "b_proj": _slice_rows(k["b_proj"], d, c),
                    "g_a": k["g_a"],
                    "g_b": _slice_rows(k["g_b"], d, c),
                    "o_norm": k["o_norm"],
                    "o_proj": _slice_cols(k["o_proj"], d, c),
                }
            else:
                s = full[l]["dsa"]
                lay["dsa"] = {
                    "q_a": s["q_a"], "q_a_ln": s["q_a_ln"],
                    "q_b": _slice_rows(s["q_b"], d, c),
                    "kv_a": s["kv_a"], "kv_a_ln": s["kv_a_ln"],
                    "kv_b": _slice_rows(s["kv_b"], d, c),
                    "o_proj": _slice_cols(s["o_proj"], d, c),
                }
            if cfg.is_moe(l):
                m = full[l]["moe"]
                sh = m["sh"]
                lay["moe"] = {
                    "gate_w": m["gate_w"], "e_bias": m["e_bias"],
                    "sh": {
                        "g": _slice_rows(sh["g"], d, c),
                        "u": _slice_rows(sh["u"], d, c),
                        "d": _slice_cols(sh["d"], d, c),
                    },
                }
            else:
                m = full[l]["mlp"]
                lay["mlp"] = {
                    "g": _slice_rows(m["g"], d, c),
                    "u": _slice_rows(m["u"], d, c),
                    "d": _slice_cols(m["d"], d, c),
                }
            chip[l] = lay
        chip["final_ln"] = np.ones(D, np.float32)
        params_by_chip.append(chip)

    embed = bf((cfg.vocab_size, D))
    lm_head = bf((cfg.vocab_size, D))
    expert_host = {}
    for l in cfg.moe_layers:
        for e in range(E):
            # fused gate|up [2I, D] with ONE fused scale grid
            gu, gu_s = u8_pair(2 * I, D)
            dd, dd_s = u8_pair(D, I)
            expert_host[(l, e)] = {
                "gu": gu, "gu_s": gu_s, "d": dd, "d_s": dd_s,
            }
    return params_by_chip, embed, lm_head, expert_host
'''
    MODS["runtime.py"] = '''"""pmap runtime for glm5_next TPU serving.

Structure (all static shapes, ~6 compiled executables):
  site_kda(p, streams, valid, rec, conv)         per KDA layer
  site_dsa(p, streams, valid, kv, bitmap, cl)    per DSA layer
  site_dense(p, streams)                          per dense-MLP layer
  site_moe(p, bank, streams [, collect_ids])      per MoE layer

- p (params), rec/conv/kv/bitmap (states), bank: per-chip shards scattered by
  pmap; streams/valid: replicated between sites.
- collect_ids: MoE site returns router ids [B,S,K] for host bank management.
- Prefill: chunks of 256 left-padded; MoE = full sweep (disjoint bank passes
  until every expert has been resident; exact for any routing).
- Decode: one token; hot banks (n_slots >= top_k * d) with exact two-phase
  refresh (snapshot state -> run -> if misses, refresh + re-run).
- Embeddings: host (numpy gather + zero-pad masking).  lm_head: host matmul.
"""
from __future__ import annotations

import threading
import time
from typing import Optional

import numpy as np

import jax
import jax.numpy as jnp
from jax import lax, pmap

from .layers import (attn_site_dsa, attn_site_kda, ffn_site_dense,
                     ffn_site_moe, moe_router, rms_norm)
from .config import GlmConfig


def _dg(a):
    return np.asarray(jax.device_get(a))


class Runner:
    # ================================================================ setup
    def __init__(self, cfg: GlmConfig, params_by_chip, embed_np, lm_head_np,
                 expert_host, log=print):
        self.cfg = cfg
        self.log = log
        self.devs = jax.devices()
        self.d = len(self.devs)
        self.expert_host = expert_host
        self.embed_np = embed_np.astype(np.float32)   # host embed (f32 copy)
        self.lm_head_np = lm_head_np.astype(np.float32)

        # ---- place dense params per chip (pmap-scattered by leading axis) ----
        # params_by_chip: list of dicts (one per chip):
        #   {l: {"attn_hc":…, "ffn_hc":…, "input_ln":…, "post_ln":…,
        #        "kda"/"dsa"/"moe"/"mlp": {...}}}
        stacked = {}   # name -> list of per-chip values (unused; kept for clarity)
        self.P = {}    # pmap-ready param pytree: {l: {site: pytree}}

        def to_jax(v):
            if isinstance(v, np.ndarray):
                return jnp.asarray(v)
            if isinstance(v, dict):
                return {k: to_jax(x) for k, x in v.items()}
            return v

        # build per-layer stacked trees: {l: {site: [chip0_val, ...chip7]}}
        self.PL = {}
        for l in range(cfg.n_layers):
            per_chip = [params_by_chip[c][l] for c in range(self.d)]
            tree = {}
            for key in per_chip[0]:
                vals = [pc[key] for pc in per_chip]
                if isinstance(vals[0], dict):
                    tree[key] = {sub: [v[sub] for v in vals] for sub in vals[0]}
                else:
                    tree[key] = vals
            self.PL[l] = tree
        # to jax arrays with leading device axis: stack leaves across chips
        def stack_tree(chip_dicts):
            return jax.tree.map(lambda *xs: jnp.stack(
                [jnp.asarray(x) for x in xs], axis=0), *chip_dicts)

        self.PJ = {l: stack_tree([params_by_chip[c][l] for c in range(self.d)])
                   for l in range(cfg.n_layers)}

        # replicated final_ln
        self.final_ln = jnp.asarray(params_by_chip[0]["final_ln"])

        # states
        self.sharding_tp = jax.sharding.NamedSharding(
            jax.sharding.Mesh(np.array(self.devs), ("tp",)),
            jax.sharding.PartitionSpec("tp"))
        self.sharding_rep = jax.sharding.NamedSharding(
            jax.sharding.Mesh(np.array(self.devs), ("tp",)),
            jax.sharding.PartitionSpec())
        self.state = self._init_state()
        self.n_slots = cfg.n_slots
        self.banks = {}          # l -> pmap-scattered bank pytree
        self.bank_ids = {}       # l -> [per-chip ids list]
        self._last_routed = {}   # l -> last chunk routed ids (np [B,S,K])
        self._init_banks()

        # compiled sites
        self._compile()

    # ------------------------------------------------------------- states
    def _init_state(self):
        """States as {key: [per-layer] -> per-chip shard list}.  Each entry
        states[key][l] is a list of d arrays (one per device) or a single
        device_put-per-device pytree; pmap scatters the stacked axis 0."""
        cfg = self.cfg
        B = 1
        nh_l = -(-cfg.n_kda_heads // self.d)
        qkvd_l = nh_l * cfg.kda_head_dim
        rec = [self._shard(np.zeros((B, nh_l, cfg.kda_head_dim,
                                     cfg.kda_head_dim), np.float32))
               for _ in range(cfg.n_layers)]
        conv = [self._shard(np.zeros((B, 3 * qkvd_l, cfg.conv_kernel - 1),
                                     np.float32))
                for _ in range(cfg.n_layers)]
        kv = [self._shard(np.zeros((B, cfg.max_ctx, cfg.kv_lora_rank),
                                   np.float32))
              for _ in range(len(cfg.dsa_layers))]
        bm = [self._shard(np.zeros((B, cfg.max_ctx), np.float32))
              for _ in range(len(cfg.dsa_layers))]
        return {"rec": rec, "conv": conv, "kv": kv, "bitmap": bm,
                "cache_len": 0}

    def _shard(self, arr):
        """Per-chip state shards stacked on axis 0, sharded P('tp') (pmap
        semantics: device i reads slice i; replication = identical slices)."""
        return jax.device_put(np.stack([arr] * self.d), self.sharding_tp)

    def _put(self, arr):
        """Replicated across chips (P())."""
        return jax.device_put(np.stack([arr] * self.d), self.sharding_rep)

    # ------------------------------------------------------------- banks
    def _bank_np(self, layer, ids_per_chip):
        I, D = self.cfg.moe_inter, self.cfg.hidden_size
        n = self.n_slots
        outs = []
        for dev_ids in ids_per_chip:
            gu = np.zeros((n, 2 * I, D), np.uint8)
            gus = np.ones((n, -(-2 * I // 128), -(-D // 128)), np.float32)
            dd = np.zeros((n, D, I), np.uint8)
            dds = np.ones((n, -(-D // 128), -(-I // 128)), np.float32)
            ids_arr = np.asarray(dev_ids, np.int32)
            for s in range(n):
                e = int(ids_arr[s]) if s < len(ids_arr) else -1
                if e < 0:
                    continue
                ex = self.expert_host[(layer, e)]
                gu[s] = ex["gu"]          # fused gate|up [2I, D]
                gus[s] = ex["gu_s"]       # fused scale grid
                dd[s] = ex["d"]
                dds[s] = ex["d_s"]
            outs.append({"gu": gu, "gu_s": gus, "d": dd, "d_s": dds,
                         "ids": ids_arr})
        return outs

    def _install_bank(self, layer, ids_per_chip):
        per_chip = self._bank_np(layer, ids_per_chip)
        # single pytree with leading device axis (like PJ params)
        self.banks[layer] = jax.device_put(
            {k: np.stack([pc[k] for pc in per_chip], axis=0)
             for k in per_chip[0]},
            self.sharding_tp)
        self.bank_ids[layer] = [list(map(int, pc["ids"])) for pc in per_chip]

    def _init_banks(self):
        for l in self.cfg.moe_layers:
            self._install_bank(l, [[-1] * self.n_slots for _ in range(self.d)])

    # ------------------------------------------------------------- compile
    def _compile(self):
        cfg = self.cfg
        self.site_kda = pmap(
            lambda p, streams, valid, rec, conv:
                attn_site_kda(p, streams, valid, rec, conv, cfg),
            axis_name="tp")
        self.site_dsa = pmap(
            lambda p, streams, valid, kv, bm, cl:
                attn_site_dsa(p, streams, valid, kv, bm, cl, cfg),
            axis_name="tp")
        self.site_dense = pmap(
            lambda p, streams: ffn_site_dense(p, streams, cfg),
            axis_name="tp")
        self._site_moe_plain = pmap(
            lambda p, bank, streams: ffn_site_moe(p, bank, streams, cfg),
            axis_name="tp")
        self._site_moe_collect = pmap(
            lambda p, bank, streams: ffn_site_moe(p, bank, streams, cfg,
                                                  collect_router=True),
            axis_name="tp")

        def _moe(p, bank, streams, collect_router=False):
            if collect_router:
                return self._site_moe_collect(p, bank, streams)
            return self._site_moe_plain(p, bank, streams)
        self.site_moe = _moe

        # final: mean over streams + final norm (single chip 0; replicated)
        self._final = jax.jit(lambda streams, w: rms_norm(
            jnp.mean(streams, axis=2), w, cfg.rms_norm_eps))

    # ------------------------------------------------------------- helpers
    def _run_attn_site(self, l, streams, valid):
        cfg = self.cfg
        p = self.PJ[l]
        if cfg.is_kda(l):
            rec = self.state["rec"][l]        # [d, B, nh_l, hd, hd]
            conv = self.state["conv"][l]
            streams, rec, conv = self.site_kda(p, streams, valid, rec, conv)
            self.state["rec"][l] = rec
            self.state["conv"][l] = conv
        else:
            di = cfg.dsa_layers.index(l)
            kv = self.state["kv"][di]
            bm = self.state["bitmap"][di]
            cl = jax.device_put(
                np.stack([np.asarray(self.state["cache_len"], np.int32)] * self.d),
                self.sharding_tp)
            streams, kv, bm = self.site_dsa(p, streams, valid, kv, bm, cl)
            self.state["kv"][di] = kv
            self.state["bitmap"][di] = bm
        return streams

    def _run_ffn_site(self, l, streams, collect_ids=False):
        cfg = self.cfg
        p = self.PJ[l]
        if cfg.is_moe(l):
            if collect_ids:
                streams, ids = self.site_moe(p, self.banks[l], streams,
                                             collect_router=True)
                return streams, ids
            return self.site_moe(p, self.banks[l], streams)
        return self.site_dense(p, streams)

    # ------------------------------------------------------------- prefill
    def reset(self):
        """Fresh recurrent/KV state (prefill starts a new sequence)."""
        self.state = self._init_state()
        self._last_routed = {}

    def prefill(self, tokens, collect_last_ids=False):
        cfg = self.cfg
        self.reset()
        S = cfg.prefill_chunk
        n = len(tokens)
        pad_total = (-n) % S if n % S else 0
        pad_total = (-n) % S
        toks = [0] * pad_total + list(tokens)
        assert len(toks) % S == 0

        last_ids = None
        for ci in range(0, len(toks), S):
            chunk = toks[ci:ci + S]
            n_real = min(S, n - (ci - pad_total))
            valid_l = [0.0] * (S - n_real) + [1.0] * n_real
            x = self.embed_np[np.asarray(chunk, np.int32)] * \
                np.asarray(valid_l, np.float32)[:, None]
            # streams: replicated [d, B, S, H, D] via identical slices P('tp')
            streams = np.broadcast_to(
                x[None, :, None, :], (1, S, cfg.hc_mult, cfg.hidden_size))
            streams = jax.device_put(
                np.stack([np.ascontiguousarray(streams, dtype=np.float32)] * self.d),
                self.sharding_tp)

            valid = jax.device_put(
                np.stack([np.asarray(valid_l, np.float32).reshape(1, S)] * self.d),
                self.sharding_tp)

            for l in range(cfg.n_layers):
                streams = self._run_attn_site(l, streams, valid)
                if l in cfg.moe_layers:
                    # exact routed-expert compute: full sweep of ALL experts
                    # in disjoint bank passes (sum over passes = exact MoE)
                    streams, ids = self._prefill_moe(l, streams)
                    self._last_routed[l] = _dg(ids[0])     # [B,S,K] chip 0
                else:
                    streams = self._run_ffn_site(l, streams)

            # final: mean + final_ln -> host
            h_last = self._final(streams, self.final_ln)   # [d, B, S, D]
            self.state["cache_len"] += n_real
            h_last_np = _dg(h_last[0, 0, -1])              # [D] last real token
            self._last_hidden = h_last_np

        self._refresh_all_banks_for_decode()
        return h_last_np

    def _prefill_moe(self, l, streams):
        """Exact MoE during prefill.

        mHC makes the site output affine in the sublayer output:
            site(y) = post * y + comb . streams      (post/comb depend only
                                                      on the INPUT streams)
        and y = shared + moe_pass.  Sweeping disjoint expert banks over
        passes p=1..P:
            sum_p site(shared + moe_p) - (P-1) * site_empty
          = P*post*shared + post*sum_p moe_p + P*(comb.streams)
            - (P-1)*(post*shared + comb.streams)
          = post*(shared + sum_p moe_p) + comb.streams
          = site(shared + ALL routed experts)         <- exact
        where site_empty runs with an empty bank (ids all -1).
        """
        cfg = self.cfg
        E = cfg.n_experts
        step = self.d * self.n_slots
        n_passes = -(-E // step)

        # empty-bank baseline (isolates hc + shared)
        self._install_bank(l, [[-1] * self.n_slots for _ in range(self.d)])
        streams_empty, _ = self._run_ffn_site(l, streams, collect_ids=True)

        total = None
        ids_out = None
        for p_i in range(n_passes):
            start = p_i * step
            ids_per_chip = []
            for dev in range(self.d):
                lo = start + dev * self.n_slots
                hi = min(lo + self.n_slots, E)
                ids_per_chip.append(list(range(lo, hi))
                                    + [-1] * (self.n_slots - max(0, hi - lo)))
            self._install_bank(l, ids_per_chip)
            streams_p, ids = self._run_ffn_site(l, streams, collect_ids=True)
            total = streams_p if total is None else total + streams_p
            ids_out = ids

        result = total - (n_passes - 1) * streams_empty
        return result, ids_out

    def _refresh_all_banks_for_decode(self):
        """Populate decode hot banks after prefill.

        Coverage target: the routed ids of the LAST real token (the next
        decode step routes exactly one token); remaining slots filled with
        the chunk's most frequent ids.  Two-phase decode guarantees exactness
        for any later miss, so no capacity error is ever raised."""
        for l in self.cfg.moe_layers:
            ids_l = self._last_routed.get(l)
            if ids_l is None:
                continue
            arr = np.asarray(ids_l).reshape(-1, self.cfg.top_k)  # [S,K]
            last = [int(i) for i in arr[-1]]
            freq = {}
            for i in arr.flatten():
                freq[int(i)] = freq.get(int(i), 0) + 1
            # priority: last token's ids first, then by frequency
            priority = list(dict.fromkeys(last))
            for i, _ in sorted(freq.items(), key=lambda kv: -kv[1]):
                if i not in priority:
                    priority.append(i)
            cap = self.d * self.n_slots
            chosen = priority[:cap]
            per_chip = [chosen[c::self.d][:self.n_slots] for c in range(self.d)]
            # pad short chips with -1
            per_chip = [pc + [-1] * (self.n_slots - len(pc)) for pc in per_chip]
            self._install_bank(l, per_chip)

    # ------------------------------------------------------------- decode
    def _decode_one(self, token, temperature, top_p):
        cfg = self.cfg
        # embed + streams (single token, always valid)
        x = self.embed_np[token]                     # [D]
        streams = np.zeros((1, 1, cfg.hc_mult, cfg.hidden_size), np.float32)
        streams[0, 0] = x[None, None, :]
        streams = jax.device_put(
            np.stack([streams] * self.d), self.sharding_tp)
        valid = jax.device_put(
            np.stack([np.ones((1, 1), np.float32)] * self.d), self.sharding_tp)

        # exact decode via bank-coverage fixpoint:
        #   run -> collected routed ids must be covered by the banks USED in
        #   that run; if not, refresh banks to cover them, roll back state,
        #   repeat.  Monotone: layer 1's router is always exact (no MoE
        #   before it), and once layers 1..j are exact they stay exact (true
        #   ids stay resident via need+old refresh), so the exact prefix
        #   grows by >= 1 per iteration -> converges in <= n_moe iterations
        #   (1-2 in practice after prefill warm-up).
        max_iters = len(cfg.moe_layers) + 1
        for it in range(max_iters):
            snap = self._snapshot_state()
            hidden, routed = self._run_all_layers(streams, valid, collect=True)
            missing = self._missing_banks(routed)
            if not missing:
                logits = self._lm_head(hidden)
                return logits
            self._refresh_hot_banks(missing)
            self._restore_state(snap)
        raise RuntimeError("decode fixpoint did not converge "
                           f"in {max_iters} iterations")

    def _run_all_layers(self, streams, valid, collect):
        routed = {}
        for l in range(self.cfg.n_layers):
            streams = self._run_attn_site(l, streams, valid)
            if self.cfg.is_moe(l) and collect:
                streams, ids = self._run_ffn_site(l, streams, collect_ids=True)
                routed[l] = _dg(ids[0])
            else:
                streams = self._run_ffn_site(l, streams)
        h = self._final(streams, self._final_ln_arg())
        return h, routed

    def _final_ln_arg(self):
        return self.final_ln

    def _missing_banks(self, routed):
        miss = {}
        for l, ids in routed.items():
            have = set()
            for pc in self.bank_ids[l]:
                have.update(e for e in pc if e >= 0)
            need = set(int(i) for i in np.asarray(ids).flatten())
            if not need <= have:
                miss[l] = need
        return miss

    def _refresh_hot_banks(self, missing):
        for l, need in missing.items():
            have = [e for pc in self.bank_ids[l] for e in pc if e >= 0]
            old = list(dict.fromkeys(have))  # LRU order preserved
            new_ids = list(dict.fromkeys(list(need) + old))[: self.d * self.n_slots]
            per_chip = [new_ids[c::self.d] for c in range(self.d)]
            self._install_bank(l, per_chip)

    def _snapshot_state(self):
        """Shallow copy: per-layer entries are immutable jax arrays, so
        replacing the list entries (not mutating) makes rollback exact."""
        return {k: (list(v) if isinstance(v, list) else v)
                for k, v in self.state.items()}

    def _restore_state(self, snap):
        self.state = {k: v for k, v in snap.items()}

    # ------------------------------------------------------------- logits
    def _lm_head(self, hidden):
        """hidden: [d, B, S, D] final-normed (or [d, B, D]); returns [V]."""
        h = _dg(hidden)
        h = h.reshape(-1, h.shape[-1])[-1]        # last row = last token
        return h @ self.lm_head_np.T              # [V]

    # ------------------------------------------------------------- gen
    def generate(self, tokens, max_new_tokens=256, temperature=0.7, top_p=0.95,
                 stop_ids=None, on_token=None):
        logits = self.prefill(tokens)
        out = []
        for i in range(max_new_tokens):
            t = self._sample(logits, temperature, top_p)
            if stop_ids and t in stop_ids:
                break
            out.append(t)
            if on_token:
                on_token(t)
            if i + 1 < max_new_tokens:
                logits = self._decode_one(t, temperature, top_p)
        return out

    def _sample(self, logits, temperature, top_p):
        v = np.asarray(logits).reshape(-1)
        if temperature <= 1e-6:
            return int(np.argmax(v))
        v = v / temperature
        v = v - v.max()
        p = np.exp(v)
        p = p / p.sum()
        if top_p and top_p < 1.0:
            order = np.argsort(-p)
            cum = np.cumsum(p[order])
            cut = np.searchsorted(cum, top_p) + 1
            keep = order[:cut]
            p2 = p[3 if False else keep]
            p2 = p[keep] / p[keep].sum()
            return int(np.random.choice(keep, p=p2))
        return int(np.random.choice(len(p), p=p))
'''
    MODS["loader_real.py"] = '''"""Real-weight loader: stream zai-org/GLM-5.3-Flash FP8 shards into host RAM
and carve them into the engine's parameter structure (mirrors make_fake()).

62 shards (~306 GiB) download in parallel (curl range stripes).  Each shard
buffer is parsed via its safetensors JSON header and carved in ONE pass with
a pre-paired (weight, scale) map: the header dict gives every tensor's
offsets up front, so pairing is trivial and order-independent.

Outputs (identical structure to params.make_fake):
  params_by_chip[c][l] = {attn_hc, ffn_hc, input_ln, post_ln,
                          kda|dsa, moe|mlp}
  embed [V,D] f32, lm_head [V,D] f32, final_ln [D] f32
  expert_host[(l, e)] = {"gu": u8[2I,D], "gu_s": f32[blocks],   # gate|up fused
                         "d": u8[D,I], "d_s": f32[blocks]}      # down

Memory: expert FP8 cold storage ~295 GiB lives in mmap'd tmpfs files (page
cache = RAM, no extra copy).  Dense params ~2.7 GiB dequantized to f32.
"""
from __future__ import annotations

import json
import os
import struct
import time
import urllib.request

import numpy as np

from .config import GlmConfig
from .fp8 import dequant_np

REPO = os.environ.get("GLM_REPO", "zai-org/GLM-5.3-Flash")
# override with the gated ab-literated variant via GLM_REPO + HF_TOKEN


def _hdrs(token=None):
    h = {"User-Agent": "glm-tpu-kernel/1.0"}
    if token:
        h["Authorization"] = f"Bearer {token}"
    return h


def shard_header(url, token=None):
    """Returns (header dict, data_start)."""
    req = urllib.request.Request(url, headers={**_hdrs(token), "Range": "bytes=0-7"})
    with urllib.request.urlopen(req, timeout=120) as r:
        hlen = struct.unpack("<Q", r.read())[0]
    req = urllib.request.Request(url, headers={**_hdrs(token),
                                               "Range": f"bytes=8-{8 + hlen - 1}"})
    with urllib.request.urlopen(req, timeout=300) as r:
        return json.loads(r.read()), 8 + hlen


def _download_shard(url, out_path, token=None, n_conn=24):
    """Parallel curl range stripes into a preallocated sparse tmpfs file.
    Returns mmap'd uint8 view."""
    import concurrent.futures as cf
    import subprocess

    req = urllib.request.Request(url, headers={**_hdrs(token), "Range": "bytes=0-0"})
    with urllib.request.urlopen(req, timeout=120) as r:
        total = int(r.headers["Content-Range"].split("/")[-1])
    if not (os.path.exists(out_path) and os.path.getsize(out_path) == total):
        with open(out_path, "wb") as f:
            f.truncate(total)
    buf = np.memmap(out_path, dtype=np.uint8, mode="r+")
    stripe = max(16 << 20, total // n_conn)
    t0 = time.time()

    def fetch(rng):
        s, e = rng
        part = f"{out_path}.part{s}"
        cmd = ["curl", "-sL", "--fail", "--retry", "4", "-r", f"{s}-{e}"]
        for k, v in _hdrs(token).items():
            cmd += ["-H", f"{k}: {v}"]
        cmd += ["-o", part, url]
        subprocess.run(cmd, check=True)
        with open(part, "rb") as f:
            buf[s:e + 1] = np.frombuffer(f.read(), dtype=np.uint8)
        os.remove(part)

    with cf.ThreadPoolExecutor(max_workers=min(n_conn, 32)) as ex:
        list(ex.map(fetch, [(s, min(s + stripe - 1, total - 1))
                            for s in range(0, total, stripe)]))
    return buf, total, time.time() - t0


# ---------------------------------------------------------------------------
# carving helpers
# ---------------------------------------------------------------------------

def _ceil(n, d):
    return -(-n // d)


def _slice_rows(w, d, c):
    chunk = _ceil(w.shape[0], d)
    return w[c * chunk:(c + 1) * chunk]


def _slice_cols(w, d, c):
    chunk = _ceil(w.shape[1], d)
    return w[:, c * chunk:(c + 1) * chunk]


def _bf16_to_f32(u16):
    return (u16.astype(np.uint32) << 16).view(np.float32)


class _Views:
    """Zero-copy tensor views into a shard buffer."""

    def __init__(self, buf, ds):
        self.buf, self.ds = buf, ds

    def u8(self, meta):
        s, e = meta["data_offsets"]
        return self.buf[self.ds + s:self.ds + e].reshape(meta["shape"])

    def f32(self, meta):
        s, e = meta["data_offsets"]
        raw = self.buf[self.ds + s:self.ds + e]
        return np.frombuffer(raw.tobytes(), np.float32).reshape(meta["shape"])

    def bf16(self, meta):
        s, e = meta["data_offsets"]
        raw = self.buf[self.ds + s:self.ds + e]
        arr = (raw.view(np.uint16).reshape(meta["shape"])
               if raw.flags["C_CONTIGUOUS"]
               else np.frombuffer(raw, np.uint16).reshape(meta["shape"]))
        return _bf16_to_f32(arr)


def carve_shard(header, views: _Views, cfg: GlmConfig, d: int, chips, expert_host,
                out):
    """Carve all tensors of one shard.  `out` dict collects embed/lm_head/
    final_ln."""
    LM = "model.language_model."

    def kda_put(l, key, w, col=False):
        for c in range(d):
            tgt = chips[c][l].setdefault("kda", {})
            tgt[key] = _slice_cols(w, d, c) if col else _slice_rows(w, d, c)

    def dsa_put(l, key, w, col=False):
        for c in range(d):
            tgt = chips[c][l].setdefault("dsa", {})
            tgt[key] = _slice_cols(w, d, c) if col else _slice_rows(w, d, c)

    # pre-pair fp8 weights with their scales
    pairs = {}
    for name, meta in header.items():
        if name == "__metadata__" or not name.endswith("weight_scale_inv"):
            continue
        wname = name[: -len("weight_scale_inv")] + "weight"
        if wname in header:
            pairs[wname] = meta

    for name, meta in header.items():
        if name == "__metadata__":
            continue
        if name.startswith("model.visual.") or ".layers.45." in name:
            continue
        if name.endswith("weight_scale_inv"):
            continue                                  # consumed via pairs

        # ---- global ----
        if name == LM + "embed_tokens.weight":
            out["embed"] = views.bf16(meta)
            continue
        if name == "lm_head.weight":
            out["lm_head"] = views.bf16(meta)
            continue
        if name == LM + "norm.weight":
            out["final_ln"] = views.bf16(meta)
            continue
        if not name.startswith(LM + "layers."):
            continue
        l = int(name[len(LM + "layers."):].split(".")[0])
        if l >= cfg.n_layers:
            continue
        tail = name[len(f"{LM}layers.{l}."):]

        # ---- replicated hc + norms ----
        if tail == "hc_attn_fn":
            for c in range(d):
                chips[c][l]["attn_hc"] = {"fn": views.bf16(meta)}
        elif tail == "hc_attn_base":
            for c in range(d):
                chips[c][l]["attn_hc"]["base"] = views.f32(meta)
        elif tail == "hc_attn_scale":
            for c in range(d):
                chips[c][l]["attn_hc"]["scale"] = views.f32(meta)
        elif tail == "hc_ffn_fn":
            for c in range(d):
                chips[c][l]["ffn_hc"] = {"fn": views.bf16(meta)}
        elif tail == "hc_ffn_base":
            for c in range(d):
                chips[c][l]["ffn_hc"]["base"] = views.f32(meta)
        elif tail == "hc_ffn_scale":
            for c in range(d):
                chips[c][l]["ffn_hc"]["scale"] = views.f32(meta)
        elif tail == "input_layernorm.weight":
            for c in range(d):
                chips[c][l]["input_ln"] = views.bf16(meta)
        elif tail == "post_attention_layernorm.weight":
            for c in range(d):
                chips[c][l]["post_ln"] = views.bf16(meta)

        # ---- KDA (all BF16) ----
        elif tail.startswith("self_attn.") and cfg.is_kda(l):
            short = tail[len("self_attn."):]
            if short.endswith(".weight"):
                short = short[:-len(".weight")]
            key = {"q_proj": "q_proj", "k_proj": "k_proj", "v_proj": "v_proj",
                   "f_a_proj": "f_a", "f_b_proj": "f_b", "b_proj": "b_proj",
                   "g_a_proj": "g_a", "g_b_proj": "g_b",
                   "o_proj": "o_proj"}.get(short.split(".")[0])
            if key is None:
                if short == "o_norm":
                    for c in range(d):
                        chips[c][l].setdefault("kda", {})["o_norm"] = views.bf16(meta)
                continue
            w = views.bf16(meta)
            if short.endswith("_conv1d"):
                w = w[:, 0, :]
            if short in ("dt_bias",):
                w = views.f32(meta)
                kda_put(l, "dt_bias", w)
            elif short == "A_log":
                w = views.f32(meta)
                kda_put(l, "A_log", w)
            elif key == "o_proj":
                kda_put(l, key, w, col=True)
            else:
                kda_put(l, key, w)

        # ---- DSA (FP8 with scales, except kv_b BF16) ----
        elif tail.startswith("self_attn.") and not cfg.is_kda(l):
            short = tail[len("self_attn."):]
            if short.endswith(".weight"):
                short = short[:-len(".weight")]
            if short.startswith("indexer"):
                continue
            if short == "q_a_layernorm":
                for c in range(d):
                    chips[c][l].setdefault("dsa", {})["q_a_ln"] = views.bf16(meta)
                continue
            if short == "kv_a_layernorm":
                for c in range(d):
                    chips[c][l].setdefault("dsa", {})["kv_a_ln"] = views.bf16(meta)
                continue
            key = {"q_a_proj": "q_a", "q_b_proj": "q_b",
                   "kv_a_proj_with_mqa": "kv_a", "kv_b_proj": "kv_b",
                   "o_proj": "o_proj"}.get(short)
            if key is None:
                continue
            smeta = pairs.get(name)
            if short == "kv_b_proj":
                w = views.bf16(meta)
            elif smeta is not None:
                w = dequant_np(views.u8(meta), views.f32(smeta)).astype(np.float32)
            else:
                w = views.bf16(meta)
            if key == "o_proj":
                dsa_put(l, key, w, col=True)
            else:
                dsa_put(l, key, w)

        # ---- MLP / MoE ----
        elif tail.startswith("mlp."):
            if tail == "mlp.gate.weight":
                w = views.bf16(meta)
                for c in range(d):
                    chips[c][l].setdefault("moe", {})["gate_w"] = w
            elif tail == "mlp.gate.e_score_correction_bias":
                w = views.f32(meta)
                for c in range(d):
                    chips[c][l].setdefault("moe", {})["e_bias"] = w
            elif tail.startswith("mlp.shared_experts."):
                short = tail[len("mlp.shared_experts."):]
                base = short[:-len(".weight")] if short.endswith(".weight") else short
                key = {"gate_proj": "g", "up_proj": "u", "down_proj": "d"}[base]
                w = dequant_np(views.u8(meta), views.f32(pairs[name])).astype(np.float32)
                sh = None
                for c in range(d):
                    tgt = chips[c][l].setdefault("moe", {}).setdefault("sh", {})
                    tgt[key] = _slice_cols(w, d, c) if key == "d" else _slice_rows(w, d, c)
            elif tail.startswith("mlp.experts."):
                # experts.{e}.{base}.weight  (scale consumed via pairs)
                parts = tail[len("mlp.experts."):].split(".")
                e, base = int(parts[0]), parts[1]
                if base not in ("gate_proj", "up_proj", "down_proj"):
                    continue
                ex = expert_host.setdefault((l, e), {})
                ex[base] = (views.u8(meta), views.f32(pairs.get(name)))
            elif l in cfg.dense_mlp_layers and tail.endswith(".weight"):
                base = tail[len("mlp."):-len(".weight")]
                if base not in ("gate_proj", "up_proj", "down_proj"):
                    continue
                key = {"gate_proj": "g", "up_proj": "u", "down_proj": "d"}[base]
                smeta = pairs.get(name)
                w = dequant_np(views.u8(meta), views.f32(smeta)).astype(np.float32) if smeta is not None else views.bf16(meta)
                for c in range(d):
                    tgt = chips[c][l].setdefault("mlp", {})
                    tgt[key] = _slice_cols(w, d, c) if key == "d" else _slice_rows(w, d, c)


def finalize_experts(expert_host, cfg, log=print):
    """Fuse gate|up into gu + gu_s (concat rows / concat scale grids)."""
    I, D = cfg.moe_inter, cfg.hidden_size
    n = 0
    for (l, e), ex in expert_host.items():
        if "gu" in ex:
            continue
        g, g_s = ex.pop("gate_proj")
        u, u_s = ex.pop("up_proj")
        ex["gu"] = np.concatenate([g, u], axis=0)
        ex["gu_s"] = np.concatenate([g_s, u_s], axis=0)
        ex["d"], ex["d_s"] = ex.pop("down_proj")
        n += 1
    log(f"[load] fused {n} experts (gu + gu_s)")


def load_real(cfg: GlmConfig, d: int, token=None, log=print,
              workdir="/dev/shm/glmw", max_shards=None):
    """Full load: returns (params_by_chip, embed, lm_head, expert_host).
    token: HF token for gated repos (orcarouter ab-literated variant)."""
    import os as _os
    if token is None:
        token = _os.environ.get("HF_TOKEN") or None
    os.makedirs(workdir, exist_ok=True)
    idx_url = f"https://huggingface.co/{REPO}/resolve/main/model.safetensors.index.json"
    with urllib.request.urlopen(urllib.request.Request(idx_url, headers=_hdrs(token)),
                                timeout=120) as r:
        index = json.loads(r.read())
    files = sorted(set(index["weight_map"].values()))
    if max_shards:
        files = files[:max_shards]
    log(f"[load] {len(files)} shards to fetch")

    chips = [{l: {"kda": None, "dsa": None, "moe": None, "mlp": None,
                  "attn_hc": None, "ffn_hc": None,
                  "input_ln": None, "post_ln": None}
              for l in range(cfg.n_layers)} for _ in range(d)]
    expert_host = {}
    out = {}

    t0 = time.time()
    for fi, fname in enumerate(files):
        url = f"https://huggingface.co/{REPO}/resolve/main/{fname}"
        out_path = os.path.join(workdir, fname)
        header, ds = shard_header(url, token)
        buf, total, dt = _download_shard(url, out_path, token)
        views = _Views(buf, ds)
        carve_shard(header, views, cfg, d, chips, expert_host, out)
        del buf
        log(f"  [{fi+1}/{len(files)}] {fname} {total/2**30:.2f} GiB "
            f"({dt:.0f}s, {total/dt/2**20:.0f} MB/s, "
            f"elapsed {(time.time()-t0)/60:.1f} min)")

    finalize_experts(expert_host, cfg, log)
    params_by_chip = []
    for c in range(d):
        chip = dict(chips[c])
        chip["final_ln"] = out["final_ln"]
        params_by_chip.append(chip)
    return params_by_chip, out["embed"], out["lm_head"], expert_host
'''
    MODS["openai_api.py"] = '''"""OpenAI-compatible API server (stdlib only) + ModelRunner interface.

Mirrors the UX of the user's GPU notebook: server on 0.0.0.0:8080, Bearer
auth, /v1/models, /v1/chat/completions (stream + non-stream), /v1/completions,
plus a /healthz probe.  Uses the tokenizers + jinja2 chat template for
formatting (chat_template.jinja from zai-org/GLM-5.3-Flash, embedded in the
notebook).
"""
from __future__ import annotations

import json
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

API_KEY = "kaggle-sfw-token-9999"
HOST = "0.0.0.0"
PORT = 8080

# The chat template (GLM 5.3) — set by the notebook from the HF repo file.
CHAT_TEMPLATE = ""
TOKENIZER = None          # tokenizers.Tokenizer instance (set by notebook)
EOS_IDS = (154820, 154827, 154829)

_model = None              # ModelRunner instance (set by serve())
_lock = threading.Lock()


class ModelRunner:
    """Interface the JAX engine plugs into."""

    def load(self) -> None: ...
    def chat(self, messages, max_tokens=512, temperature=0.7, top_p=0.95,
             stream_cb=None, reasoning_effort=None) -> str: ...
    def reset(self) -> None: ...


def _render_chat(messages, reasoning_effort=None):
    from jinja2.sandbox import ImmutableSandboxedEnvironment
    env = ImmutableSandboxedEnvironment(
        trim_blocks=True, lstrip_blocks=True,
        extensions=["jinja2.ext.loopcontrols"])
    tmpl = env.from_string(CHAT_TEMPLATE)
    return tmpl.render(messages=messages,
                      add_generation_prompt=True,
                      reasoning_effort=reasoning_effort,
                      tokenize=False)


def _extract_reply(text):
    """Strip reasoning blocks and take the final answer segment."""
    # GLM: <think>...</think> then the reply; split on the LAST </think>
    if "</think>" in text:
        text = text.split("</think>")[-1]
    return text.strip()


class Handler(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def log_message(self, fmt, *args):
        pass  # keep the kernel log clean

    def _auth(self):
        h = self.headers.get("Authorization", "")
        return h == f"Bearer {API_KEY}"

    def _send_json(self, code, obj):
        body = json.dumps(obj).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        if self.path == "/healthz":
            self._send_json(200, {"ok": True})
        elif self.path == "/v1/models":
            self._send_json(200, {"object": "list", "data": [
                {"id": "glm-5.3-flash-uncensored-fp8", "object": "model",
                 "owned_by": "local"}]})
        else:
            self._send_json(404, {"error": "not found"})

    def do_POST(self):
        if not self._auth():
            self._send_json(401, {"error": "bad api key"})
            return
        n = int(self.headers.get("Content-Length", 0))
        body = json.loads(self.rfile.read(n) or b"{}")
        if self.path == "/v1/chat/completions":
            self._chat(body)
        elif self.path == "/v1/completions":
            self._completion(body)
        else:
            self._send_json(404, {"error": "not found"})

    def _chat(self, body):
        messages = body.get("messages", [])
        max_tokens = int(body.get("max_tokens", 512))
        temperature = float(body.get("temperature", 0.7))
        top_p = float(body.get("top_p", 0.95))
        stream = bool(body.get("stream", False))
        effort = body.get("reasoning_effort")

        prompt = _render_chat(messages, effort)
        t0 = time.time()
        with _lock:
            if stream:
                self._stream_chat(prompt, max_tokens, temperature, top_p, t0)
                return
            text = _model.chat(messages, max_tokens=max_tokens,
                               temperature=temperature, top_p=top_p)
        dt = time.time() - t0
        reply = _extract_reply(text)
        self._send_json(200, {
            "id": "chatcmpl-local", "object": "chat.completion",
            "created": int(time.time()), "model": "glm-5.3-flash-uncensored-fp8",
            "choices": [{"index": 0, "finish_reason": "stop",
                         "message": {"role": "assistant", "content": reply}}],
            "usage": {"prompt_tokens": 0, "completion_tokens": len(reply.split()),
                      "total_tokens": len(reply.split())},
            "_timing_s": round(dt, 2),
        })

    def _stream_chat(self, prompt, max_tokens, temperature, top_p, t0):
        self.send_response(200)
        self.send_header("Content-Type", "text/event-stream")
        self.send_header("Cache-Control", "no-cache")
        self.end_headers()

        def cb(tok_text, idx):
            chunk = {"id": "chatcmpl-local", "object": "chat.completion.chunk",
                     "created": int(time.time()),
                     "model": "glm-5.3-flash-uncensored-fp8",
                     "choices": [{"index": 0,
                                  "delta": {"content": tok_text}}]}
            try:
                self.wfile.write(f"data: {json.dumps(chunk)}\n\n".encode())
                self.wfile.flush()
            except Exception:
                pass

        text = _model.chat(prompt if isinstance(prompt, list) else
                           [{"role": "user", "content": prompt}],
                           max_tokens=max_tokens,
                           temperature=temperature, top_p=top_p,
                           stream_cb=cb)
        # final chunk + done
        done = {"id": "chatcmpl-local", "object": "chat.completion.chunk",
                "choices": [{"index": 0, "delta": {}, "finish_reason": "stop"}]}
        try:
            self.wfile.write(f"data: {json.dumps(done)}\n\n".encode())
            self.wfile.write(b"data: [DONE]\n\n")
            self.wfile.flush()
        except Exception:
            pass

    def _completion(self, body):
        prompt = body.get("prompt", "")
        if isinstance(prompt, list):
            prompt = "\n".join(p if isinstance(p, str) else p.get("text", "")
                               for p in prompt)
        max_tokens = int(body.get("max_tokens", 256))
        with _model_lock():
            text = _model.chat([{"role": "user", "content": prompt}],
                               max_tokens=max_tokens)
        self._send_json(200, {
            "id": "cmpl-local", "object": "text_completion",
            "created": int(time.time()), "model": "glm-5.3-flash-uncensored-fp8",
            "choices": [{"text": text, "index": 0, "finish_reason": "stop"}],
        })


def _model_lock():
    return _lock


def serve(model, port=None, host=None):
    """Start the blocking HTTP server with the given ModelRunner."""
    global _model, PORT, HOST, CHAT_TEMPLATE, TOKENIZER, EOS_IDS
    _model = model
    if port:
        PORT = port
    if host:
        HOST = host
    httpd = ThreadingHTTPServer((HOST, PORT), Handler)
    print(f"[server] listening on http://{HOST}:{PORT}/v1  (key: {API_KEY[:8]}...)", flush=True)
    httpd.serve_forever()
'''
    MODS["runner_glue.py"] = '''"""ModelRunner: binds the JAX engine + tokenizer + chat template to the
OpenAI API server.  This is the last glue module; the notebook embeds it."""
from __future__ import annotations

import threading
import time

import numpy as np

from .config import GlmConfig
from .runtime import Runner


class GlmModelRunner:
    """openai_api.ModelRunner implementation over the pmap Runner."""

    def __init__(self, runner: Runner, tokenizer, chat_template: str,
                 log=print):
        self.r = runner
        self.tok = tokenizer
        self.chat_template = chat_template
        self.log = log
        self.eos = set(runner.cfg.eos_ids)

    # ------------------------------------------------------------------
    def _render(self, messages, reasoning_effort=None):
        from jinja2.sandbox import ImmutableSandboxedEnvironment
        env = ImmutableSandboxedEnvironment(
            trim_blocks=True, lstrip_blocks=True,
            extensions=["jinja2.ext.loopcontrols"])   # {% break %} support
        tmpl = env.from_string(self.chat_template)
        return tmpl.render(messages=messages, add_generation_prompt=True,
                           reasoning_effort=reasoning_effort, tokenize=False)

    def chat(self, messages, max_tokens=512, temperature=0.7, top_p=0.95,
             stream_cb=None, reasoning_effort=None):
        prompt = self._render(messages, reasoning_effort)
        enc = self.tok.encode(prompt, add_special_tokens=False)
        ids = list(enc.ids) if hasattr(enc, "ids") else list(enc)
        if len(ids) > self.r.cfg.max_ctx - max_tokens - 8:
            ids = ids[-(self.r.cfg.max_ctx - max_tokens - 8):]

        t0 = time.time()
        logits = self.r.prefill(ids)
        n_pref = len(ids)
        out_ids = []
        det_text = ""
        for i in range(max_tokens):
            t = self.r._sample(logits, temperature, top_p)
            if t in self.eos:
                break
            out_ids.append(t)
            if stream_cb and (i + 1) % 4 == 0:
                piece = self.tok.decode(out_ids)
                if piece != det_text:
                    stream_cb(piece[len(det_text):], i)
                    det_text = piece
            if i + 1 < max_tokens:
                logits = self.r._decode_one(t, temperature, top_p)
        text = self.tok.decode(out_ids)
        dt = time.time() - t0
        self.log(f"[gen] prefill {n_pref} tok + {len(out_ids)} gen tok "
                 f"in {dt:.1f}s ({len(out_ids)/max(dt-0.01,0.01):.1f} tok/s)")
        return text

    def reset(self):
        self.r.reset()
'''
    MODS["__init__.py"] = ""

    os.makedirs("/kaggle/tmp/glmtpu", exist_ok=True)
    for name, src in MODS.items():
        with open(f"/kaggle/tmp/glmtpu/{name}", "w", encoding="utf-8") as f:
            f.write(src)
    sys.path.insert(0, "/kaggle/tmp")

from glmtpu.config import GlmConfig
import glmtpu.layers, glmtpu.runtime, glmtpu.loader_real
print("engine modules ready; loader repo =", glmtpu.loader_real.REPO)

In [ ]:
# ---- engine self-test on the real TPU (fake weights, tiny config) ----
import numpy as np
from glmtpu.config import GlmConfig
from glmtpu.params import make_fake
from glmtpu.runtime import Runner

cfg = GlmConfig.tiny()
pbc, embed, lm_head, expert_host = make_fake(cfg, d=8, seed=1)
r = Runner(cfg, pbc, embed, lm_head, expert_host)
tokens = np.random.randint(0, cfg.vocab_size, size=100).tolist()
g = r.generate(tokens, max_new_tokens=8, temperature=0.0)
g2 = r.generate(tokens, max_new_tokens=8, temperature=0.0)
assert g == g2, "greedy determinism failed"
print("TPU self-test OK:", g)

In [ ]:
# ---- tokenizer + chat template ----
import urllib.request

BASE = "https://huggingface.co/zai-org/GLM-5.3-Flash/resolve/main"
for fn in ["tokenizer.json", "tokenizer_config.json", "chat_template.jinja"]:
    urllib.request.urlretrieve(f"{BASE}/{fn}", f"/kaggle/tmp/{fn}")
    print("got", fn)

from tokenizers import Tokenizer
tok = Tokenizer.from_file("/kaggle/tmp/tokenizer.json")
chat_template = open("/kaggle/tmp/chat_template.jinja").read()
print("vocab:", tok.get_vocab_size())

In [ ]:
# ---- load real weights: 62 shards (~306 GiB) -> host RAM ----
import numpy as np
from glmtpu.config import GlmConfig
from glmtpu.loader_real import load_real
from glmtpu.runtime import Runner

t0 = time.time()
cfg = GlmConfig.real()
cfg.n_slots = 8          # hot experts per chip per MoE layer in HBM
params_by_chip, embed, lm_head, expert_host = load_real(
    cfg, d=8, log=print, workdir="/dev/shm/glmw")
print(f"weights loaded in {(time.time()-t0)/60:.1f} min; experts: {len(expert_host)}")

runner = Runner(cfg, params_by_chip, embed, lm_head, expert_host, log=print)
print("runner ready")
del params_by_chip

In [ ]:
# ---- generation sanity check ----
prompt = "[gMASK]<sop><|user|>\nSay something with teeth. Be brief.\n<|assistant|>\n<think>\n"
ids = tok.encode(prompt, add_special_tokens=False)
print("prompt tokens:", len(ids))
t0 = time.time()
logits = runner.prefill(ids)
out = []
for i in range(64):
    t = runner._sample(logits, 0.7, 0.95)
    if t in runner.cfg.eos_ids:
        break
    out.append(t)
    logits = runner._decode_one(t, 0.7, 0.95)
print(f"generated {len(out)} tokens in {time.time()-t0:.1f}s")
print("OUTPUT:", tok.decode(out)[:500])

In [ ]:
# ---- OpenAI-compatible server + public cloudflared tunnel ----
import threading, subprocess, re, time

from glmtpu import openai_api
from glmtpu.runner_glue import GlmModelRunner

model = GlmModelRunner(runner, tok, chat_template)
server = threading.Thread(target=openai_api.serve,
                          args=(model, 8080, "0.0.0.0"), daemon=True)
server.start()
time.sleep(2)

cf = "/kaggle/tmp/cloudflared"
if not os.path.exists(cf):
    subprocess.run(["wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", cf], check=True)
    subprocess.run(["chmod", "+x", cf], check=True)
subprocess.Popen([cf, "tunnel", "--url", "http://localhost:8080",
                  "--no-autoupdate"],
                 stdout=open("/kaggle/tmp/tunnel.log", "w"),
                 stderr=subprocess.STDOUT)
url = None
for _ in range(24):
    time.sleep(5)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",
                  open("/kaggle/tmp/tunnel.log").read())
    if m:
        url = m.group(0)
        break
assert url, "no tunnel URL"
print("=" * 60)
print("PUBLIC API:", url + "/v1")
print("API KEY:   kaggle-sfw-token-9999")
print("=" * 60)
with open("/kaggle/working/api_url.txt", "w") as f:
    f.write(url + "/v1\n")

In [ ]:
# ---- smoke test through the public URL (openai client) ----
import subprocess
subprocess.run(["pip", "install", "-q", "openai"], check=False)
import re
from openai import OpenAI
url = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",
                open("/kaggle/tmp/tunnel.log").read()).group(0)
c = OpenAI(base_url=f"{url}/v1", api_key="kaggle-sfw-token-9999")
r = c.chat.completions.create(
    model="glm-5.3-flash-uncensored-fp8",
    messages=[{"role": "user", "content": "Say something with teeth."}],
    max_tokens=256, temperature=0.7)
reply = r.choices[0].message.content
print(reply)
with open("/kaggle/working/api_smoke.txt", "w") as f:
    f.write(reply or "")

In [ ]:
# ---- keep alive until the session ends ----
import time
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    pass